# Week 3, day 2 — Graded 04 SOLUTIONS: Open problems   (tier 4 of 4)

Executed in the lab image (pandas 3.0.5) against the real files in `../data/`.
Every quoted number is what it actually printed.

These are **one** defensible answer each. Where a different choice would give a
different number, the note says so and gives that number too.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Graded 04 — Open problems. Run this once.
import numpy as np
import pandas as pd

orders = pd.read_csv("../data/orders_long.csv")
cust = pd.read_csv("../data/customers_messy.csv")

print("orders:", orders.shape, "| customers:", cust.shape)
print("years:", sorted(orders["Year"].unique()))

TIER 4 — state the figure, the denominator, and what would move it

### Question 1

By percentage growth 2009->2012: **Nunavut `+728.5%`**, NWT `+39.9%`, Ontario `+18.9%`, then five declines. By absolute change: **Ontario `+23771.68`**, the largest by far.

Two defensible answers and they disagree completely, which is the point.

**Nunavut is the trap.** It grew 728% -- on `1251.32` in 2009 rising to
`10367.59`, across **9 orders in the entire four-year file**. One large order
in 2012 produced that percentage. It is a real number and it is not a trend.

Ontario is the answer worth giving: `+23771.68` on a base of `125806.91`,
over 302 orders. Smaller percentage, and it is the only one you could plan
around.

**Figure**: Ontario, +18.9%, +23,772. **Denominator**: 302 orders across
four years. **What would change it**: using percentage growth instead of
absolute, which puts a 9-order region first.

In [ ]:
pt = orders.pivot_table(index="Region", columns="Year",
                        values="Sales", aggfunc="sum")
growth = pd.DataFrame({
    "2009": pt[2009].round(2),
    "2012": pt[2012].round(2),
})
growth["change"] = (growth["2012"] - growth["2009"]).round(2)
growth["pct"] = ((growth["2012"] / growth["2009"] - 1) * 100).round(1)
print(growth.sort_values("pct", ascending=False).to_string())
print()
print("orders behind each region:")
print(orders.groupby("Region").size().to_string())

### Question 2

mean **`1468.96`**, median **`404.91`**, modal band `100-500`, skew `3.74`.

Give the **median**, `404.91`, and say so.

The mean is 3.6 times the median because the distribution has a long right
tail -- skew 3.74. 'Typical' means the order in the middle, and half of all
orders are under 405 while the mean sits at 1,469 because of a few very
large ones.

Quoting the mean is not wrong, it just answers a different question: total
revenue divided by order count, which is the right number for capacity
planning and the wrong one for 'what does a customer usually spend'.

**Figure**: 404.91. **Denominator**: all 1,093 orders. **What would change
it**: quoting the mean, which more than triples it.

In [ ]:
s = orders["Sales"]
print("mean:   %.2f" % s.mean())
print("median: %.2f" % s.median())
print("mode band:", pd.cut(s, bins=[0, 100, 500, 2000, float("inf")],
                           labels=["<100", "100-500", "500-2000", "2000+"])
      .value_counts().idxmax())
print()
print(s.describe().round(2).to_string())
print()
print("skew: %.2f" % s.skew())

### Question 3

Mean profit split by whether a discount was applied, per category, with order counts alongside.

The comparison to make is *within* category, because the categories have
very different order sizes -- comparing discounted Furniture against
undiscounted Office Supplies confounds the two effects.

Be careful with the causal claim. Discounts are not applied at random:
they go to large orders, competitive deals, and slow-moving stock. A
category where discounted orders are less profitable may be one where
discounting hurts, or one where the hard-to-sell items get discounted. This
data cannot separate those.

**Figure**: whichever category shows the widest gap. **Denominator**: the
order counts printed beside it. **What would change it**: controlling for
order size, which is the obvious confounder.

In [ ]:
work = orders.copy()
work["Discounted"] = work["Discount"] > 0

g = work.groupby(["Category", "Discounted"]).agg(
    orders=("OrderID", "count"),
    mean_profit=("Profit", "mean"),
    total_profit=("Profit", "sum"),
).round(2)
print(g.to_string())
print()
print("mean discount by category:")
print(work.groupby("Category")["Discount"].mean().round(4).to_string())

### Question 4

**`88.8%`** -- `355` complete rows of `400`, after cleaning. `Province` is the only incomplete column, also at `88.8%`.

The figure depends entirely on what you did first, and that is the answer
to the question rather than a caveat.

Here: markers replaced with `NaN`, names stripped, exact duplicates removed.
Skip the marker replacement and completeness reads 96%, because `-` and `?`
count as values. Skip the de-duplication and the denominator is 440.

**Figure**: 88.8%. **Denominator**: 400 de-duplicated customer rows.
**What would change it**: not treating `-` and `?` as missing, which would
report 96%.

In [ ]:
work = cust.copy()
work["Province"] = work["Province"].replace(["-", "?"], np.nan)
work["CustomerName"] = work["CustomerName"].str.strip()
work = work.drop_duplicates()

per_col = (work.notna().sum() / len(work) * 100).round(1)
print("per-column completeness (%):")
print(per_col.to_string())
print()
complete_rows = work.dropna()
print("rows with no missing value: %d of %d (%.1f%%)"
      % (len(complete_rows), len(work), 100 * len(complete_rows) / len(work)))
print("distinct customers:", work["CustomerID"].nunique())

### Question 5

Monthly share of annual revenue, against the `8.3%` a perfectly flat year would give.

Some months are meaningfully above and below the flat line, so there is
variation -- but this is **four years pooled into twelve buckets**, so a
single exceptional month in a single year moves its bucket permanently.

A real seasonality claim needs the same month compared across years:
`groupby(["Year", "Month"])`, then look at whether the pattern repeats. If
December is high in all four years that is seasonality; if it is high in one,
that is an event.

**Figure**: the spread between the lowest and highest month. **Denominator**:
1,093 orders pooled across four years. **What would change it**: separating
the years, which is the only way to tell a pattern from an incident.

In [ ]:
by_month = orders.groupby("Month").agg(
    orders=("OrderID", "count"),
    sales=("Sales", "sum"),
).round(2)
by_month["pct_of_year"] = (100 * by_month["sales"] / by_month["sales"].sum()).round(1)
print(by_month.to_string())
print()
print("if perfectly flat each month would be %.1f%%" % (100 / 12))
print("spread: %.1f%% (lowest) to %.1f%% (highest)"
      % (by_month["pct_of_year"].min(), by_month["pct_of_year"].max()))

### Question 6

Top **10%** of orders carry **`53.2%`** of revenue; top 20% carry `73.5%`; top 50% carry `95.1%`. Top 3 regions carry `65.3%`.

Steeper than the usual 80/20: the top fifth of orders is nearly three
quarters of the money, and the bottom half is 5%.

That single fact reframes several other questions in this folder. It is why
the IQR fence removes 57% of revenue for 11% of rows (graded 03 Q9), why the
mean order is 3.6x the median, and why a category's total tells you almost
nothing without its order count.

**Figure**: 53.2% from the top 10% of orders. **Denominator**: 1,093 orders.
**What would change it**: concentrating by customer or product rather than
by order, which would almost certainly be steeper again.

In [ ]:
s = orders["Sales"].sort_values(ascending=False)
cum = s.cumsum() / s.sum()
for pct in (0.1, 0.2, 0.5):
    n = int(len(s) * pct)
    print("top %3d%% of orders (%4d) carry %.1f%% of revenue"
          % (pct * 100, n, 100 * cum.iloc[n - 1]))
print()
by_region = orders.groupby("Region")["Sales"].sum().sort_values(ascending=False)
print("top 3 regions carry %.1f%% of revenue"
      % (100 * by_region.head(3).sum() / by_region.sum()))

### Question 7

A `Region` x `Category` table of sums **and counts**, with margins.

The counts are the reason this table is defensible and a sums-only version
is not.

Nunavut has 9 orders in the whole file. Without the count beside it, its
row reads like every other region's and any per-region comparison silently
treats 9 observations as equivalent to 302. Every misleading number in tiers
3 and 4 comes back to a denominator someone did not print.

Region down the side because a region is something a person owns; category
across because there are only three of them and a wide table with three
columns is readable.

In [ ]:
pt = orders.pivot_table(index="Region", columns="Category", values="Sales",
                        aggfunc=["sum", "count"], margins=True)
print(pt.round(0).to_string())

# Region down the side because that is who owns a number; category across
# because there are only three. Counts alongside sums so nobody reads a
# large total from a handful of orders as a trend -- Nunavut has 9 orders
# in the whole file.

### Question 8

`orders["Region"].mean()` -> **raises** `TypeError: Cannot perform reduction 'mean' with string dtype`.

It is different in kind because it is **not a question about the data at
all** -- it is a category error. There is no ordering on region names, so
there is no middle one, and no amount of choosing a better statistic fixes
that.

Every other question in this tier had several defensible answers and the
work was choosing among them. This one has none, and Pandas is right to
refuse.

Which is the note to end four tiers on. Pandas will stop you when a
question is *meaningless*. It will not stop you when a question is
reasonable and your answer to it is wrong -- and that is nearly every
question you will actually be asked.

In [ ]:
print("Region dtype:", orders["Region"].dtype)
print("distinct values:", orders["Region"].nunique())
print(orders["Region"].mean())